# Cloudscene Country Counts — World Bank Country-Driven Scraper

**Date:** 2025-08-19

This notebook loads a **World Bank population CSV** to get the country list, then attempts to fetch
**Cloudscene** per-country data-centre counts for those countries. It supports both:
- a simple `requests + BeautifulSoup` fetcher (static HTML parsing), and
- an optional Selenium fetcher for dynamic pages (if needed).

**Ethics & ToS:** Please review Cloudscene's Terms of Service and robots.txt. Use polite rate limiting.
If ToS disallow automated access, collect a manual snapshot and use the builder notebook instead.


## 0) Install dependencies (uncomment if needed)

In [ ]:

# %pip install pandas pyarrow numpy requests beautifulsoup4 lxml python-slugify
# Optional (only if dynamic rendering is required):
# %pip install selenium webdriver-manager


## 1) Config — paths, URL patterns, selectors, rate limits

In [ ]:

import os, time, random, re, math
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import pandas as pd
import numpy as np
from slugify import slugify

# === INPUTS ===
WORLD_BANK_POP_CSV = "/mnt/data/world_pop_long.csv"  # or world_pop_long.parquet if you prefer
OUTPUT_PARQUET      = "/mnt/data/global_dc_counts_tidy.parquet"
SNAPSHOT_YEAR       = 2024

# If you only have the Parquet:
# import pandas as pd
# df_pop = pd.read_parquet("/mnt/data/world_pop_long.parquet")
# df_pop[['country','iso3']].drop_duplicates().to_csv("/mnt/data/world_pop_countries.csv", index=False)

# === SCRAPER CONFIG ===
# Base URL patterns to try per country; update once you inspect Cloudscene's structure in your browser.
URL_PATTERNS = [
    "https://cloudscene.com/country/{slug}",                     # example placeholder
    "https://cloudscene.com/browse/data-centers/{slug}",         # example placeholder
    "https://cloudscene.com/search/data-centers?country={query}" # example placeholder
]

# CSS selectors or regex patterns to extract the integer count from the HTML.
# The code will try each extractor in order until one returns a number.
EXTRACTORS = [
    # ("css", "span[data-testid='country-dc-count']::text"),  # CSS example (pseudo)
    ("regex", r"(\d{1,4})\s*(?:Data\s*Centers|Data\s*Centres)"),   # text pattern like "523 Data Centers"
    ("regex", r"Data\s*Centers:\s*(\d{1,4})"),
    ("regex", r"Total\s*Data\s*Centers\s*:?[\s<>]*(\d{1,4})"),
]

# polite rate limiting
REQUESTS_PER_MIN = 20
BASE_SLEEP = 60.0 / REQUESTS_PER_MIN  # base seconds between requests
JITTER = 0.3                           # +/-30% jitter

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0; +your-email@example.com)"
}

# Manual overrides for tricky country names -> slug or query
OVERRIDE_SLUGS = {
    "Côte d'Ivoire": "cote-divoire",
    "Congo, Dem. Rep.": "democratic-republic-of-the-congo",
    "Congo, Rep.": "republic-of-the-congo",
    "United States": "united-states",
    "United Kingdom": "united-kingdom",
    "South Korea": "south-korea",
    "North Korea": "north-korea",
    "Eswatini": "swaziland",
    "Türkiye": "turkey",
}

# dry-run: set to True to print URLs without fetching
DRY_RUN = True


## 2) Load World Bank countries

In [ ]:

# Expecting a long format with columns like: country, iso3, year, population
df_pop = pd.read_csv(WORLD_BANK_POP_CSV)
cols = [c.lower() for c in df_pop.columns]
df_pop.columns = cols
if "country" not in df_pop.columns:
    raise ValueError("World Bank CSV must contain a 'country' column.")

countries = (df_pop[["country"]].drop_duplicates().assign(country=lambda d: d["country"].astype(str).str.strip()))
print("Unique countries:", len(countries))

# If you have an iso3 column, keep it for later joins
if "iso3" in df_pop.columns:
    countries = (df_pop[["country","iso3"]]
                 .drop_duplicates()
                 .assign(country=lambda d: d["country"].astype(str).str.strip()))
countries.head(10)


## 3) Build candidate country URLs

In [ ]:

def country_slug(name: str) -> str:
    # slugify and apply overrides
    name = name.strip()
    if name in OVERRIDE_SLUGS:
        return OVERRIDE_SLUGS[name]
    # Basic cleanups
    return slugify(name)

def candidate_urls(name: str) -> List[str]:
    slug = country_slug(name)
    query = name.replace(" ", "+")
    return [pat.format(slug=slug, query=query) for pat in URL_PATTERNS]

countries["slug"] = countries["country"].apply(country_slug)
countries["candidate_urls"] = countries["country"].apply(candidate_urls)
countries.head(5)


## 4) Fetchers — requests + BeautifulSoup (and optional Selenium)

In [ ]:

import requests
from bs4 import BeautifulSoup

def sleep_politely():
    t = BASE_SLEEP * (1 + np.random.uniform(-JITTER, JITTER))
    t = max(0.5, t)
    time.sleep(t)

def extract_count_from_html(html: str) -> Optional[int]:
    # Try regex extractors first (robust across templates)
    for kind, patt in EXTRACTORS:
        if kind == "regex":
            m = re.search(patt, html, flags=re.IGNORECASE)
            if m:
                try:
                    val = int(m.group(1).replace(",", ""))
                    return val
                except Exception:
                    pass
    # Then try basic BeautifulSoup approaches (extend with known selectors)
    soup = BeautifulSoup(html, "lxml")
    # Example: look for any element containing "Data Centers" followed by a number
    texts = soup.find_all(text=re.compile(r"Data\s*Cent(er|re)s", re.I))
    for t in texts:
        window = t.parent.get_text(" ", strip=True) if t and t.parent else str(t)
        m = re.search(r"(\d{1,4})", window)
        if m:
            try: return int(m.group(1))
            except: pass
    return None

def fetch_count_requests(url: str) -> Tuple[Optional[int], Optional[str]]:
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}"
        cnt = extract_count_from_html(r.text)
        return cnt, None
    except Exception as e:
        return None, f"ERR {e}"

# Optional Selenium path (uncomment if needed)
USE_SELENIUM = False

if USE_SELENIUM:
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service as ChromeService
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    def fetch_count_selenium(url: str) -> Tuple[Optional[int], Optional[str]]:
        try:
            options = webdriver.ChromeOptions()
            options.add_argument("--headless=new")
            options.add_argument("--no-sandbox")
            options.add_argument("--disable-gpu")
            driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
            driver.set_page_load_timeout(60)
            driver.get(url)
            # Wait for main content to load (tune selector)
            WebDriverWait(driver, 20).until(lambda d: d.page_source and "Data" in d.page_source)
            html = driver.page_source
            cnt = extract_count_from_html(html)
            driver.quit()
            return cnt, None
        except Exception as e:
            return None, f"SELENIUM ERR {e}"


## 5) Run extraction (dry-run by default)

In [ ]:

rows = []
for i, row in countries.iterrows():
    name = row["country"]
    urls = row["candidate_urls"]
    iso3 = row["iso3"] if "iso3" in row else None

    dc_count = None
    source_url = None
    status = "not_fetched"

    for url in urls:
        if DRY_RUN:
            print(f"[DRY-RUN] Would fetch: {url}")
            status = "dry_run"
            source_url = url
            break
        # live fetch
        if USE_SELENIUM:
            cnt, err = fetch_count_selenium(url)
        else:
            cnt, err = fetch_count_requests(url)
        if cnt is not None:
            dc_count = cnt
            source_url = url
            status = "ok"
            break
        else:
            status = err or "no_count_found"
            sleep_politely()

    rows.append({"country": name, "iso3": iso3, "dc_count": dc_count, "source_url": source_url, "status": status})

df_out = pd.DataFrame(rows)
df_out.head(12)


## 6) Post-process, zero-fill, and write Parquet

In [ ]:

# Zero-fill missing counts (countries Cloudscene may not list)
df_out["dc_count"] = df_out["dc_count"].fillna(0).astype(int)

# Add snapshot year
df_out["snapshot_year"] = SNAPSHOT_YEAR

# Write tidy Parquet
os.makedirs(os.path.dirname(OUTPUT_PARQUET) or ".", exist_ok=True)
df_out[["country","dc_count","iso3","snapshot_year","source_url","status"]].to_parquet(OUTPUT_PARQUET, index=False)
print(f"Wrote {len(df_out)} rows → {OUTPUT_PARQUET}")
df_out["status"].value_counts(), df_out.head(10)


## 7) Next steps
- Set `DRY_RUN = False` and adjust `URL_PATTERNS` and `EXTRACTORS` **after inspecting Cloudscene in your browser**.
- If pages are dynamic, set `USE_SELENIUM = True` and tune the wait/selector logic.
- Respect Cloudscene’s ToS and add longer sleeps (`REQUESTS_PER_MIN`) if needed.
- If automated access is disallowed, capture a **manual CSV snapshot** and use the *Counts Builder* notebook instead.
